# 01 — Data Preparation

This notebook runs the **stdlib-only data pipeline** (`src/data/prepare.py`) end to end on CPU:

1. **Load** the raw seed file (`data/raw/sample_seed.jsonl`, see `raw_data_path` in `config.yaml`)
2. **Clean / normalize** records (strip whitespace, normalize language tags, drop empties)
3. **Deduplicate** near-duplicates at the configured similarity threshold (`0.85`)
4. **Split** into train / validation / test (15% val, 15% test, seeded shuffle with `seed: 42`)

Every parameter comes from `config.yaml` via `src/data/config_loader.py` — nothing is hard-coded.

In [ ]:
import sys
from pathlib import Path

# Repo root = directory containing config.yaml
ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.yaml").exists(), "run this notebook from the repo root"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

## Load configuration (single source of truth)

We read the data-pipeline paths and split ratios straight from `config.yaml`.

In [ ]:
import json

from src.data.config_loader import load_config

cfg = load_config(str(ROOT / "config.yaml"))
print(json.dumps(
    {k: cfg[k] for k in (
        "dataset_version", "seed", "raw_data_path", "processed_data_path",
        "train_path", "validation_path", "test_path",
        "test_size", "val_size", "dedup_similarity_threshold",
    )},
    indent=2,
))

## Run the stdlib data pipeline

This executes `src.data.prepare` (CPU, no installs needed). It requires the raw seed file
`data/raw/sample_seed.jsonl` to exist — if it is missing you will get a clear message below
instead of a silent empty run.

In [ ]:
import subprocess

import src.data.prepare as prepare

raw = ROOT / cfg["raw_data_path"]
if not raw.exists():
    print(f"RAW DATA MISSING: {raw}")
    print("Place the seed JSONL at data/raw/sample_seed.jsonl, then re-run this cell.")
else:
    entry = getattr(prepare, "main", None)
    if entry is not None:
        stats = entry()  # module entry point; reads paths from config.yaml
        print("pipeline returned:", stats)
    else:
        # fallback: run as a module (uses the __main__ guard)
        subprocess.run([sys.executable, "-m", "src.data.prepare"], cwd=ROOT, check=True)
    print("\ndata pipeline finished.")

## Dataset statistics

Count records in each split produced by the pipeline.

In [ ]:
def count_jsonl(path):
    p = ROOT / path
    if not p.exists():
        return f"MISSING ({path})"
    with p.open(encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

for name, key in [("train", "train_path"), ("validation", "validation_path"), ("test", "test_path")]:
    print(f"{name:>10}: {count_jsonl(cfg[key])} records  <- {cfg[key]}")
print(f"{'cleaned':>10}: {count_jsonl(cfg['processed_data_path'])} records  <- {cfg['processed_data_path']}")